# 📜 Translations

This notebook explains how translations are sourced, how to download them, and how to use them in the parsing pipeline.

**Important:** Translations come from the ORACC website and represent scholarly interpretations of the texts. They are stored separately from the cuneiform data and are completely unaffected by parsing configuration options (`max_break_fraction`, `drop_missing`, `drop_damaged`). This notebook explains what that means in practice.

## 🎯 Goals

1. **Understand the source** — where translations come from and what they contain
2. **Download the translation cache** — fetch pre-cached translations from Zenodo
3. **Parse with translations** — enable translations in the pipeline and access them
4. **Understand the relationship to parsing config** — why translations can show text that the cuneiform output does not

In [2]:
# Install oracc-parser from PyPI
!pip install oracc-parser -q

# --- Optional: persist downloaded data across Colab sessions using Google Drive ---
# Without this, data downloads to /content/oracc_data and is lost when the session ends.
# Uncomment the lines below to mount your Drive and store data there persistently.
#
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ["ORACC_DATA_DIR"] = "/content/drive/MyDrive/oracc_data"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 3.6 MB/s eta 0:00:00
Mounted at /content/drive


## 1. What are translations?

For many ORACC projects, scholars have produced primarily English (more rarely German) translations that are published alongside the cuneiform text on the ORACC website. The parser fetches these translations by downloading the HTML page for each tablet and extracting the translation lines.

A few things to keep in mind:

- **Not all tablets have translations.** Coverage varies by project — some projects are fully translated, others have none. Tablets without a translation return an empty string.
- **Translations are scholarly reconstructions.** They reflect the editor's interpretation of the full text, including damaged or missing sections. A tablet that is 30% broken may still have a complete translation.
- **Translations are not stored in the word CSVs.** They are fetched separately and stored in `enriched_data/cache/html/`. There is no alignment between the cuneiform texts and their translations.

## 2. Getting the translation data

There are two ways to get translations.

### Option A — Download the pre-cached archive from Zenodo (recommended)

The Zenodo record includes `oracc_html_translations.zip`, a 130 MB archive of pre-downloaded HTML pages for all 25,994 tablets in the dataset. Download it once and all subsequent translation lookups are instant (no network requests).

In [4]:
from oracc_parser.download.fetch_data import fetch_data
from oracc_parser.settings import TRANSLATIONS_DIR

sentinel = TRANSLATIONS_DIR / ".translations_complete"

if sentinel.exists():
    n_cached = len(list(TRANSLATIONS_DIR.rglob("*.html")))
    print(f"Translation cache complete: {n_cached} HTML pages cached")
else:
    print("Downloading full translation cache from Zenodo (~130 MB)...")
    fetch_data(include_translations=True)
    n_cached = len(list(TRANSLATIONS_DIR.rglob("*.html")))
    print(f"Done: {n_cached} HTML pages now cached")


Fetching file list from https://zenodo.org/records/20625379...
  oracc_html_translations.zip (130.0 MB)
  catalogues.zip (2.7 MB)


oracc_html_translations.zip: 100%|██████████| 136M/136M [00:16<00:00, 8.15MB/s]


  ✓ catalogues.zip already extracted, skipping

📂 Extracting oracc_html_translations.zip ...


Extracting HTML translations: 100%|██████████| 25903/25903 [06:57<00:00, 62.07file/s]


✅ Extracted 25903 files to /content/drive/MyDrive/oracc_data/cache/html

✓ Done. Data is ready in /content/drive/MyDrive/oracc_data
Done: 25903 HTML pages now cached


### Option B — Fetch live from ORACC (for projects not in Zenodo)

If you are working with a project that is not in the Zenodo dataset, or if you add a new project yourself (see notebook 04), translations can be fetched automatically from the ORACC website the first time each tablet is parsed. The HTML page is then saved to the local cache, so subsequent runs are instant.

This happens when `fetch_translations=True` in `RunConfig` (the default is `False`). No extra steps are needed, but be aware that parsing a large project for the first time will make one HTTP request per tablet.

## 3. Parsing with translations

Set `fetch_translations=True` in `RunConfig`. With the Zenodo cache present, this reads from disk and adds no noticeable overhead.

In [5]:
from oracc_parser import parse_project_from_word_csvs, RunConfig
from oracc_parser.io.word_csv import load_word_csvs_from_dir
from oracc_parser.settings import WORD_CSV_DIR

PROJECT = "borsippa"
LIMIT = 5

project_csv_dir = WORD_CSV_DIR / PROJECT.replace("/", "-")
word_dfs = load_word_csvs_from_dir(project_csv_dir, project=PROJECT)
if LIMIT:
    word_dfs = dict(list(word_dfs.items())[:LIMIT])

config = RunConfig(fetch_translations=True)
records = parse_project_from_word_csvs(PROJECT, word_dfs, config=config)

n_with_translation = sum(1 for r in records if r.content.english_translation)
print(f"Parsed {len(records)} tablets — {n_with_translation} have an English translation")

12:53:36 | INFO    | oracc_parser | Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/borsippa
INFO:oracc_parser:Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/borsippa
Processing borsippa:   0%|          | 0/5 [00:00<?, ?it/s]12:53:36 | ERROR   | oracc_parser | Error loading language qcu-949: 1 validation error for Language
normalized_language
  Input should be 'Sumerian', 'Proto-cuneiform', 'Akkadian', 'Ugaritic', 'Urartian', 'Persian', 'Hittite', 'Aramaic', 'Elamite', 'Hurrian', 'Canaanite', 'Numbers', 'Unclear', 'Proto-Elamite', 'Greek' or 'Egyptian' [type=literal_error, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.12/v/literal_error
ERROR:oracc_parser:Error loading language qcu-949: 1 validation error for Language
normalized_language
  Input should be 'Sumerian', 'Proto-cuneiform', 'Akkadian', 'Ugaritic', 'Urartian', 'Persian', 'Hittite', 'Aramaic', 'Elamite', 'Hurrian', 'Can

Parsed 5 tablets — 5 have an English translation


In [6]:
# Find a tablet that has a translation and print it
for record in records:
    if record.content.english_translation:
        print(f"=== {record.metadata.id_text} ===")
        print()
        print("TRANSLITERATION (first 300 chars):")
        trans = record.content.transliterated_str_representation
        print(trans.text[:300] if trans else "(none)")
        print()
        print("ENGLISH TRANSLATION:")
        print(record.content.english_translation)
        break

=== P202111 ===

TRANSLITERATION (first 300 chars):
2 1/2 u₄-mu maš-ti-ti ša₂ {iti}GUD 2 1/2 u₄-mu
[maš]-ti-ti {iti}KIN 2 1/2 u₄-mu kap-ri ša₂ {iti}GAN
1 u₄]-mu maš-ti-ti ša₂ {iti}ZIZ₂ 2 u₄-mu maš-ti-ti ša₂ {iti}ŠE
[PAB] 10 1/2 u₄-mu kap-ri u₃ maš-ti-ti
[i]-⸢na⸣ GIŠ.ŠUB.BA ša₂ {m}{d}EN-SUM{na} DUMU-šu₂ ša₂ {m}{d}AG-EDURU-MU DUMU {m}DINGIR-ia₂
[a-di] 

ENGLISH TRANSLATION:
2½ days maštītu in month II
2½ days maštītu (in) month VI
2½ days kapru in month IX
[1] day maštītu in month XI
2 days maštītu in month XII
[in total] 10½ days kapru and maštītu [of (?)] the prebend of Bēl-iddin/Nabû-aplu-iddin/Ilia: (these days) are at the disposal of Marduk-šumu-ibni for performing (ana ēpišānūti) until the end of the agreement.
Marduk-šumu-ibni [shall measure out (barley and dates)] from the bīt-asê.
[He is responsible for] approaching the meal, [for uninterrupted service and punctuality]. [ (...) ]
Marduk-šumu-ibni has not received [ x ] [fro]m Bēl-iddin.
Witnesses: Nabû-uballiṭ/Nabû-šumu-iddin/Il

## 4. Translations are independent of parsing configuration

The `RunConfig` options that control how broken or damaged signs are handled — `max_break_fraction`, `drop_missing`, and `drop_damaged` — operate on the **cuneiform data only**. They affect the transliteration, normalization, lemmatization, and Unicode cuneiform outputs. The `english_translation` field is always the full scholarly text, unmodified.

This means:

| Config option | What it affects | Translation affected? |
|---|---|---|
| `max_break_fraction=0.0` | Replaces any word with ≥1 broken sign with `X` in transliteration/normalization/lemmatization | ❌ No |
| `drop_missing=True` | Removes `[x]` signs from the Unicode cuneiform output | ❌ No |
| `drop_damaged=True` | Removes `⸢x⸣` signs from the Unicode cuneiform output | ❌ No |
| `mask_pos=["PN"]` | Replaces personal name lemmas with a mask token | ❌ No |

**Practical consequence:** if you set `max_break_fraction=0.0` to work only with fully intact words, you will see `X` tokens in the transliteration wherever words were damaged — but the translation will still contains the scholarly interpretation of the broken segments.

In [7]:
# Demonstrate: strict break filtering vs. full translation
config_strict = RunConfig(fetch_translations=True, max_break_fraction=0.0)
records_strict = parse_project_from_word_csvs(PROJECT, word_dfs, config=config_strict)

# Pick a tablet that has both a translation and some damage
for r_default, r_strict in zip(records, records_strict):
    trans_default = r_default.content.transliterated_str_representation
    trans_strict  = r_strict.content.transliterated_str_representation
    if not r_default.content.english_translation:
        continue
    if trans_default and trans_strict and trans_default.text != trans_strict.text:
        print(f"=== {r_default.metadata.id_text} ===")
        print()
        print("TRANSLITERATION (default config — all words kept):")
        print(trans_default.text[:400])
        print()
        print("TRANSLITERATION (max_break_fraction=0.0 — damaged words replaced with X):")
        print(trans_strict.text[:400])
        print()
        print("ENGLISH TRANSLATION (same in both configs):")
        print(r_default.content.english_translation)
        print()
        assert r_default.content.english_translation == r_strict.content.english_translation
        print("✓ Translation is identical regardless of max_break_fraction")
        break

Processing borsippa: 100%|██████████| 5/5 [00:00<00:00,  6.01it/s]
12:53:57 | INFO    | oracc_parser | Processed 5 tablets for borsippa from word CSVs
INFO:oracc_parser:Processed 5 tablets for borsippa from word CSVs


=== P202111 ===

TRANSLITERATION (default config — all words kept):
2 1/2 u₄-mu maš-ti-ti ša₂ {iti}GUD 2 1/2 u₄-mu
[maš]-ti-ti {iti}KIN 2 1/2 u₄-mu kap-ri ša₂ {iti}GAN
1 u₄]-mu maš-ti-ti ša₂ {iti}ZIZ₂ 2 u₄-mu maš-ti-ti ša₂ {iti}ŠE
[PAB] 10 1/2 u₄-mu kap-ri u₃ maš-ti-ti
[i]-⸢na⸣ GIŠ.ŠUB.BA ša₂ {m}{d}EN-SUM{na} DUMU-šu₂ ša₂ {m}{d}AG-EDURU-MU DUMU {m}DINGIR-ia₂
[a-di] ṭup-pi-šu₂ a-na e-piš-ša₂-nu-tu i-na pa-ni
{[m}{d]}⸢AMAR⸣.UTU-MU-ib-ni DUMU-šu₂ ša₂ {m}šu-la-a DUMU 

TRANSLITERATION (max_break_fraction=0.0 — damaged words replaced with X):
2 1/2 u₄-mu maš-ti-ti ša₂ {iti}GUD 2 1/2 u₄-mu
X {iti}KIN 2 1/2 u₄-mu kap-ri ša₂ {iti}GAN
X X maš-ti-ti ša₂ {iti}ZIZ₂ 2 u₄-mu maš-ti-ti ša₂ {iti}ŠE
X 10 1/2 u₄-mu kap-ri u₃ maš-ti-ti
X GIŠ.ŠUB.BA ša₂ {m}{d}EN-SUM{na} DUMU-šu₂ ša₂ {m}{d}AG-EDURU-MU DUMU {m}DINGIR-ia₂
X ṭup-pi-šu₂ a-na e-piš-ša₂-nu-tu i-na pa-ni
X DUMU-šu₂ ša₂ {m}šu-la-a DUMU {m}DINGIR-ia₂
X X X {m}{d}AMAR.UTU-MU-ib-ni TA e-su-u₂
X 

ENGLISH TRANSLATION (same in both configs):
2½ days ma

## 5. Working with translations at scale

For large-scale analysis, use the flat helper functions rather than iterating over records manually.

In [8]:
from oracc_parser import get_translations, get_full_flat_table

# Translation-only table
trans_df = get_translations(records)
print(f"Translation table: {trans_df.shape[0]} rows")
display(trans_df)

Translation table: 5 rows


,id,project,translation
0,borsippa_P202111,borsippa,2½ days maštītu in month II\n2½ days maštītu (...
1,borsippa_P202351,borsippa,[Šulā son of] Ṣillā of the Ilia family has [vo...
2,borsippa_P202504,borsippa,[x da]ys tardennu (offering) from [day x until...
3,borsippa_P202510,borsippa,20 š silver of the yield of his prebend of the...
4,borsippa_P202948,borsippa,"0;3.4 barley, the price of prime quality billa..."


In [9]:
# Filter to tablets that actually have a translation
has_translation = trans_df[trans_df["translation"] != ""]
print(f"{len(has_translation)}/{len(trans_df)} tablets have an English translation in this sample")

5/5 tablets have an English translation in this sample


In [10]:
# Full flat table — transliteration, normalization, lemmatization, unicode, and translation in one DataFrame
flat = get_full_flat_table(records)
print(f"Columns: {list(flat.columns)}")
display(flat.head())

Columns: ['id', 'project', 'text_id', 'genre', 'archive', 'provenance', 'period', 'start_year', 'end_year', 'transliteration', 'normalization', 'lemmatization', 'unicode', 'translation', 'total_tokens', 'tokens_without_broken']


,id,project,text_id,genre,archive,provenance,period,start_year,end_year,transliteration,normalization,lemmatization,unicode,translation,total_tokens,tokens_without_broken
0,borsippa_P202111,borsippa,P202111,Legal,Ilia Archive,Borsippa,achaemenid,-547,-331,2 1/2 u₄-mu maš-ti-ti ša₂ {iti}GUD 2 1/2 u₄-mu...,UNK UNK ūmu maštīti ša Ayyaru UNK UNK ūmu\nmaš...,UNK UNK ūmu maštītu ša Ayyaru UNK UNK ūmu\nmaš...,𒈫 𒈦 𒌓𒈬 𒈦𒋾𒋾 𒃻 𒌗𒄞 𒈫 𒈦 𒌓𒈬\n𒈦𒋾𒋾 𒌗𒆥 𒈫 𒈦 𒌓𒈬 𒆏𒊑 𒃻 𒌗𒃶\...,2½ days maštītu in month II\n2½ days maštītu (...,102,101
1,borsippa_P202351,borsippa,P202351,Legal,Ilia Archive,Borsippa,Neo-Babylonian,-626,-539,{[m}šu-la-a A-šu₂ ša₂] {m}ṣil-la-a A {m}DINGIR...,Šula māršu ša Ṣilla mār Ilia\nina hūd libbišu ...,Šula māru ša Ṣilla māru Ilia\nina hūdu libbu K...,𒁹𒋗𒆷𒀀 𒀀𒋙 𒃻 𒁹𒉣𒆷𒀀 𒀀 𒁹𒀭𒅀\n𒀸 𒄷𒌓 𒊮𒁉𒋙 𒁹𒆗𒁀𒀀 𒇽𒃲𒆷𒋙\n𒀀𒈾 𒇽...,[Šulā son of] Ṣillā of the Ilia family has [vo...,157,148
2,borsippa_P202504,borsippa,P202504,Legal,Ahiya’ūtu Archive,Borsippa,achaemenid,-547,-331,[x u₄]-mu tar-din-nu {iti}ŠE TA UD [x]-KAM\n[a...,UNK ūmu tardinnu Addaru ultu ūm UNK\nadi ūm UN...,UNK ūmu tardennu Addaru ištu ūmu UNK\nadi ūmu ...,X 𒌓𒈬 𒋻𒁷𒉡 𒌗𒊺 𒋫 𒌓 X𒄰\n𒀀𒁲 𒌓 𒌋𒐋𒄰 𒐈 𒌓𒈬 𒋾𒅆𒆕𒈨𒌍\n𒌗𒁈 𒌓 ...,[x da]ys tardennu (offering) from [day x until...,247,231
3,borsippa_P202510,borsippa,P202510,Legal,Ibnāya Archive,Borsippa,achaemenid,-547,-331,1/3 GIN₂ KU₃.BABBAR ina BURU₁₄ GIŠ.ŠUB.BA-šu₂\...,UNK šiqil kaspi ina ebūr isqišu\nša šanat UNK ...,UNK šiqlu kaspu ina ebūru isqu\nša šattu UNK C...,𒑚 𒂆 𒆬𒌓 𒀸 𒂙 𒄑𒊒𒁀𒋙\n𒃻 𒈬 𒁹𒄰 𒁹𒅗𒄠𒁍𒍣𒅀\n𒈗 𒁷𒌁𒆠 𒌉𒋙 𒃻 𒁹𒆪𒊏...,20 š silver of the yield of his prebend of the...,96,96
4,borsippa_P202948,borsippa,P202948,Legal,Ilia Archive,Borsippa,Neo-Babylonian,-626,-539,3 (PI) 4 BAN₂ ŠE.BAR ŠAM₂ KAŠ.U₂.<SA> SIG₅\nna...,UNK pān UNK sūt uṭṭati šīm billati damqi\nnapt...,UNK pānu UNK sūtu uṭṭatu šīmu billatu damqu\nn...,𒐈 𒉿 𒐉 𒑏 𒊺𒁇 𒉚 𒁉𒌑𒊓 𒅆𒂟\n𒀮𒋫𒉡 𒃻 𒀭𒉺 𒃻 𒁹𒀭𒀝𒁾𒆰\n𒀀𒋙 𒃻 𒁹𒀭...,"0;3.4 barley, the price of prime quality billa...",66,66


## What's next?

- **Quickstart:** See `01_quickstart.ipynb` to download data, parse a project from CSVs, and export results.
- **Configure parsing:** See `03_configure_and_export.ipynb` for masking PNs, dropping broken signs, and changing output formats.
- **Understand the process:** See `04_oracc_json_processing.ipynb` for how the package processes raw ORACC JSON files and how to add new projects.